# Evidence Retrieval — Patch: Save Raw Text Evidence

**Run this as a standalone notebook** (fresh kernel, NOT continuing from evidence_retrieval.ipynb).

**What it does:** The original `evidence_retrieval.ipynb` saved only numeric features.
This patch rebuilds the FAISS index from scratch and saves proper
`raw_evidence_*.pkl` files with actual caption text + retrieved caption strings.

**Before running:** Delete these files if they exist:
- `cache/stance_scores/stance_raw_all_v2.pkl`
- `cache/stance_scores/stance_raw_all_v3.pkl`

## 0. Imports & Paths

In [1]:
import os, sys, json, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
import faiss

try:
    ROOT = Path(__file__).resolve().parents[2]
except NameError:
    ROOT = Path(os.getcwd())
    while not (ROOT / 'src' / 'utils' / 'paths.py').exists():
        ROOT = ROOT.parent
        if ROOT == ROOT.parent:
            ROOT = Path(os.getcwd()); break

sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

CLIP_EMB_DIR = ROOT / 'cache' / 'clip_embeddings'
EVIDENCE_DIR = ROOT / 'cache' / 'evidence'
STANCE_DIR   = ROOT / 'cache' / 'stance_scores'
NC_META_DIR  = ROOT / 'data'  / 'newsclippings' / 'metadata'
LORA2_DIR    = ROOT / 'cache' / 'lora_stage2'
STANCE_DIR.mkdir(parents=True, exist_ok=True)

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def save_pkl(obj, path):
    path = Path(path)
    with open(path, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    print(f"  Saved {path.name} ({path.stat().st_size/1e6:.1f} MB)")

# Delete old stance caches now so we don't forget
for old in ('stance_raw_all.pkl', 'stance_raw_all_v2.pkl', 'stance_raw_all_v3.pkl'):
    p = STANCE_DIR / old
    if p.exists():
        p.unlink()
        print(f"Deleted old cache: {old}")

print("Setup done.")

Project root: f:\GitHub\eve-rag
Setup done.


## 1. Load CLIP Embeddings

In [2]:
print("Loading CLIP embeddings …")
clip_img = {s: load_pkl(CLIP_EMB_DIR / 'clip_image_embeddings' / f'{s}.pkl')
            for s in ('train', 'val', 'test')}
clip_txt = {s: load_pkl(CLIP_EMB_DIR / 'clip_text_embeddings' / f'{s}.pkl')
            for s in ('train', 'val', 'test')}
for s in ('train', 'val', 'test'):
    print(f"  {s}: img={len(clip_img[s])}  txt={len(clip_txt[s])}")

# Verify key type is int
sample_key = next(iter(clip_img['test']))
print(f"Key type: {type(sample_key).__name__} ({'✅ int' if isinstance(sample_key, int) else '❌'})")

Loading CLIP embeddings …
  train: img=473663  txt=385850
  val: img=50000  txt=40864
  test: img=50000  txt=40915
Key type: int (✅ int)


## 2. Load NC JSONs & Metadata

In [3]:
def load_nc_json(path):
    """Load NC JSON — always returns a plain list of annotation dicts."""
    with open(path) as f:
        raw = json.load(f)
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        # Root-level JSONs: {'annotations': [...], 'source_datasets': {...}}
        if 'annotations' in raw:
            return raw['annotations']
        # Fallback: first list value
        for v in raw.values():
            if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict):
                return v
    raise ValueError(f"Cannot parse NC JSON at {path}")


print("Loading NC JSONs …")
nc_json = {
    'train': load_nc_json(ROOT / 'data' / 'newsclippings' / 'train.json'),
    'val':   load_nc_json(ROOT / 'data' / 'newsclippings' / 'data' / 'merged_balanced' / 'val.json'),
    'test':  load_nc_json(ROOT / 'data' / 'newsclippings' / 'test.json'),
}
for split, rows in nc_json.items():
    print(f"  {split}: {len(rows)} pairs  keys={list(rows[0].keys())}")

print("\nLoading NC metadata (captions) …")
nc_meta = {}
for split in ('train', 'val', 'test'):
    with open(NC_META_DIR / f'{split}.json') as f:
        entries = json.load(f)
    if isinstance(entries, list):
        nc_meta[split] = {int(e['id']): e.get('caption', '') for e in entries}
    else:
        nc_meta[split] = {
            int(k): v.get('caption', '') if isinstance(v, dict) else str(v)
            for k, v in entries.items()
        }
    print(f"  {split}: {len(nc_meta[split])} captions")

Loading NC JSONs …
  train: 71072 pairs  keys=['id', 'image_id', 'similarity_score', 'source_dataset', 'falsified']
  val: 7024 pairs  keys=['id', 'image_id', 'similarity_score', 'source_dataset', 'falsified']
  test: 7264 pairs  keys=['id', 'image_id', 'similarity_score', 'source_dataset', 'falsified']

Loading NC metadata (captions) …
  train: 385003 captions
  val: 40830 captions
  test: 40915 captions


## 3. Build FAISS Index & Reverse Map

In [4]:
def to_float32(v):
    arr = np.array(v, dtype=np.float32)
    if arr.ndim > 1: arr = arr.flatten()[:512]
    return arr


print("Building FAISS index on training image embeddings …")
train_img_ids = sorted(clip_img['train'].keys())   # list of int image_ids
train_img_matrix = np.vstack([
    to_float32(clip_img['train'][iid]) for iid in tqdm(train_img_ids, desc='stacking')
]).astype(np.float32)
faiss.normalize_L2(train_img_matrix)
index = faiss.IndexFlatIP(train_img_matrix.shape[1])
index.add(train_img_matrix)
print(f"  FAISS index: {index.ntotal} vectors, dim={train_img_matrix.shape[1]}")

# Reverse map: image_id → list of {pair_id, falsified}
train_imgid_to_pairs = {}
for row in nc_json['train']:
    iid = int(row['image_id'])
    pid = int(row['id'])
    fal = int(row['falsified'])
    train_imgid_to_pairs.setdefault(iid, []).append({'pair_id': pid, 'falsified': fal})
print(f"  train_imgid_to_pairs: {len(train_imgid_to_pairs)} unique images")

K = 10

Building FAISS index on training image embeddings …


stacking:   0%|          | 0/473663 [00:00<?, ?it/s]

  FAISS index: 473663 vectors, dim=512
  train_imgid_to_pairs: 59876 unique images


## 4. Text-Aware Retrieval Function

In [5]:
def get_caption(split, pair_id):
    """Look up caption from metadata. Falls back to train metadata."""
    cap = nc_meta.get(split, {}).get(int(pair_id), '')
    if not cap:
        cap = nc_meta['train'].get(int(pair_id), '')
    return str(cap)


def retrieve_with_text(query_img_emb, query_img_id, query_caption, k=K):
    """FAISS retrieval returning list of {text, score, nb_img_id, nb_pair_id}."""
    q_img  = to_float32(query_img_emb)
    q_norm = (q_img / (np.linalg.norm(q_img) + 1e-8)).reshape(1, -1).astype(np.float32)

    sims, idxs = index.search(q_norm, k + 1)
    sims = sims[0]; idxs = idxs[0]

    retrieved  = []
    seen_texts = {query_caption}

    for sim, pos in zip(sims, idxs):
        if pos < 0 or pos >= len(train_img_ids): continue
        nb_img_id = train_img_ids[pos]
        if nb_img_id == int(query_img_id): continue   # skip self

        for p in train_imgid_to_pairs.get(nb_img_id, [])[:2]:
            cap_text = get_caption('train', p['pair_id'])
            if not cap_text or cap_text in seen_texts: continue
            seen_texts.add(cap_text)
            retrieved.append({
                'text':       cap_text,
                'score':      float(sim),
                'nb_img_id':  int(nb_img_id),
                'nb_pair_id': int(p['pair_id']),
            })
        if len(retrieved) >= k: break

    return retrieved


# Smoke test
_row   = nc_json['test'][0]
_iid   = int(_row['image_id'])
_pid   = int(_row['id'])
_cap   = get_caption('test', _pid)
_q_img = to_float32(clip_img['test'].get(_iid, np.zeros(512)))
_ret   = retrieve_with_text(_q_img, _iid, _cap, k=3)
print(f"Smoke test — caption: {_cap[:70]}")
for r in _ret:
    print(f"  score={r['score']:.3f} | {r['text'][:70]}")
assert len(_ret) > 0, "❌ Retrieval returned nothing"
assert 'text' in _ret[0], "❌ Retrieved item has no 'text' field"
print("✅ Retrieval working correctly")

Smoke test — caption: Flashmob protest at Crossrail site in Central London by Unite union me
  score=0.703 | A SWAT officer peers around a corner as his team searches neighborhood
✅ Retrieval working correctly


## 5. Build & Save NC Raw Evidence

In [ ]:
print("Building raw text evidence for NC splits …")

for split in ('train', 'val', 'test'):
    raw_text_evidence = {}

    for row in tqdm(nc_json[split], desc=f'NC {split}'):
        img_id   = int(row['image_id'])
        pair_id  = int(row['id'])
        false    = int(row['falsified'])
        pair_key = f"{pair_id}_{false}"
        caption  = get_caption(split, pair_id)

        q_img_raw = clip_img[split].get(img_id)
        retrieved = retrieve_with_text(q_img_raw, img_id, caption, k=K) \
                    if q_img_raw is not None else []

        raw_text_evidence[pair_key] = {
            'caption':   caption,
            'retrieved': retrieved,
            'label':     false,
            'pair_key':  pair_key,
            'image_id':  img_id,
            'pair_id':   pair_id,
        }

    n_empty_cap = sum(1 for v in raw_text_evidence.values() if not v['caption'])
    n_empty_ret = sum(1 for v in raw_text_evidence.values() if not v['retrieved'])
    sample = next(iter(raw_text_evidence.values()))
    print(f"  {split}: {len(raw_text_evidence)} pairs | "
          f"empty cap: {n_empty_cap} | empty ret: {n_empty_ret}")
    print(f"    caption:      {sample['caption'][:70]}")
    if sample['retrieved']:
        print(f"    retrieved[0]: {sample['retrieved'][0]['text'][:70]}")

    save_pkl(raw_text_evidence, STANCE_DIR / f'raw_evidence_{split}.pkl')

print("\n✅ NC raw evidence saved.")

Building raw text evidence for NC splits …


NC train:   0%|          | 0/71072 [00:00<?, ?it/s]

## 6. Build & Save OOD Raw Evidence (VERITE + MMFakeBench)

In [ ]:
VERITE_CSV  = ROOT / 'data' / 'verite' / 'image-text-verification' / 'VERITE' / 'VERITE.csv'
MMFAKE_JSON = ROOT / 'data' / 'mmfakebench' / 'MMFakeBench_test.json'
LABEL_MAP   = {'true': 0, 'miscaptioned': 1, 'out-of-context': 1}

mm_embs     = load_pkl(LORA2_DIR / 'ood_embs_mmfake.pkl')
verite_embs = load_pkl(LORA2_DIR / 'ood_embs_verite.pkl')
print(f"OOD embs — VERITE: {len(verite_embs)}, MMFake: {len(mm_embs)}")

df_ver      = pd.read_csv(VERITE_CSV)
verite_caps = df_ver['caption'].tolist()
verite_lbls = [LABEL_MAP.get(str(r).lower().strip(), 0) for r in df_ver['label'].tolist()]

with open(MMFAKE_JSON) as f:
    mmfake_data = json.load(f)
mmfake_pairs = mmfake_data if isinstance(mmfake_data, list) else list(mmfake_data.values())
mmfake_caps  = [str(r.get('text', r.get('caption', ''))) for r in mmfake_pairs]
mmfake_lbls  = [1 if 'fake' in str(r.get('gt_answers', r.get('label','real'))).lower()
                else 0 for r in mmfake_pairs]

print(f"VERITE: {len(verite_caps)} captions | MMFake: {len(mmfake_caps)} captions")

In [ ]:
def build_ood_raw_evidence(emb_dict, captions, labels, desc='OOD'):
    results = {}
    for i, (key, entry) in enumerate(tqdm(emb_dict.items(), desc=desc)):
        caption = captions[i] if i < len(captions) else ''
        label   = labels[i]   if i < len(labels)   else int(entry.get('label', 0))

        # Clip emb may be 768-d — take first 512 for FAISS
        q_img = np.array(entry['clip_img_emb'], dtype=np.float32).flatten()
        if len(q_img) < 512:
            q_img = np.pad(q_img, (0, 512 - len(q_img)))
        else:
            q_img = q_img[:512]

        # Use img_id=-1 so no self-match is skipped
        retrieved = retrieve_with_text(q_img, -1, str(caption), k=K)

        results[i] = {
            'caption':   str(caption),
            'retrieved': retrieved,
            'label':     int(label),
            'pair_key':  i,
        }

    n_empty = sum(1 for v in results.values() if not v['retrieved'])
    print(f"  {desc}: {len(results)} pairs | empty retrieved: {n_empty}")
    sample = next(iter(results.values()))
    print(f"    caption:      {sample['caption'][:70]}")
    if sample['retrieved']:
        print(f"    retrieved[0]: {sample['retrieved'][0]['text'][:70]}")
    return results


print("Building VERITE raw evidence …")
save_pkl(build_ood_raw_evidence(verite_embs, verite_caps, verite_lbls, 'VERITE'),
         STANCE_DIR / 'raw_evidence_verite.pkl')

print("Building MMFakeBench raw evidence …")
save_pkl(build_ood_raw_evidence(mm_embs, mmfake_caps, mmfake_lbls, 'MMFake'),
         STANCE_DIR / 'raw_evidence_mmfake.pkl')

print("\n✅ OOD raw evidence saved.")

## 7. Verify All Files

In [ ]:
print("Final verification …\n")
all_ok = True
for name in ('raw_evidence_train.pkl', 'raw_evidence_val.pkl', 'raw_evidence_test.pkl',
             'raw_evidence_verite.pkl', 'raw_evidence_mmfake.pkl'):
    p = STANCE_DIR / name
    if not p.exists():
        print(f"  ❌ MISSING: {name}"); all_ok = False; continue
    d      = load_pkl(p)
    sample = next(iter(d.values()))
    has_cap = bool(sample.get('caption', ''))
    has_ret = bool(sample.get('retrieved', []))
    ret_ok  = has_ret and isinstance(sample['retrieved'][0], dict) and 'text' in sample['retrieved'][0]
    ok = has_cap and ret_ok
    if not ok: all_ok = False
    print(f"  {'✅' if ok else '❌'} {name:35s} {len(d):6d} pairs | "
          f"cap={'✅' if has_cap else '❌'} ret={'✅' if ret_ok else '❌'}")
    print(f"      caption:      {sample['caption'][:70]}")
    if has_ret:
        print(f"      retrieved[0]: {sample['retrieved'][0]['text'][:70]}")
    print()

if all_ok:
    print("✅ All raw evidence files verified. Run stance_scoring.ipynb now.")
else:
    print("❌ Some files missing or malformed — check errors above.")